# 18
Побудувати часовий ряд виручки по днях, застосувати rolling-середнє (вікно 7), а потім альтернативно порахувати те саме через NumPy (cumsum) і порівняти максимальну різницю між методами.

#### У чому суть завдання
Суть завдання полягає в тому, щоб проаналізувати великий масив даних (у нашому випадку це згенеровані 100 000 записів щоденної виручки) та згладити щоденні стрибки, обчисливши ковзне середнє за 7 днів. Це дозволяє побачити загальний тренд замість хаотичного "паркану" щоденних продажів. Додатковою і головною метою є порівняння двох різних підходів до цього розрахунку: вбудованого методу бібліотеки Pandas та математичного трюку за допомогою NumPy, щоб перевірити їхню точність та заміряти, який з них працює швидше на великому обсязі даних.

#### Як працює перший спосіб
Перший спосіб використовує стандартний інструмент бібліотеки Pandas — функцію rolling(window=7).mean(). Вона працює максимально прямолінійно: бере перші сім днів, додає їхню виручку, ділить на сім і записує результат; потім зміщує це "вікно" на один день вперед і повторює операцію заново для нових семи днів. Цей підхід дуже зручний, читабельний і простий у написанні, але під капотом комп'ютер робить багато зайвої роботи, щоразу перераховуючи суму тих самих чисел, які вже були в попередньому вікні.

#### Як працює другий спосіб
Другий спосіб спирається на масиви NumPy та математичну хитрість із кумулятивною (накопичувальною) сумою — cumsum. Замість того, щоб щоразу по черзі додавати 7 чисел, алгоритм один раз рахує наростаючий підсумок для всього ряду від самого початку. Далі, щоб дізнатися суму за будь-які 7 днів, він просто бере загальний підсумок на кінець цього тижня і віднімає від нього підсумок, який був до початку цього тижня. Завдяки цьому замість семи операцій додавання комп'ютер виконує лише одну операцію віднімання на кожен крок. Це робить такий алгоритм неймовірно швидким і ефективним.

#### Трохи про те, як працює сам код
Програма написана дуже професійно з використанням об'єктно-орієнтованого підходу: уся логіка акуратно зібрана в клас RevenueAnalyzer. Особливістю цього коду є система вимірювання часу: він використовує декораторний патерн _measure_execution та високоточний таймер time.perf_counter(), щоб засікати, скільки мілісекунд виконується кожна функція. Результати часу елегантно зберігаються за допомогою сучасного інструменту @dataclass. У блоці if __name__ == "__main__": програма створює 100 000 тестових рядків, "нацьковує" на них обидва алгоритми, переконується, що максимальна похибка між ними майже нульова, і роздруковує звіт про швидкість.

In [2]:
import pandas as pd
import numpy as np
import time
from dataclasses import dataclass
from typing import Callable

@dataclass
class ExecutionTime:
    method_name: str
    duration: float
    
    def __str__(self) -> str:
        return f"{self.method_name}: {self.duration:.6f}s"

class RevenueAnalyzer:
    
    def __init__(self, data: pd.DataFrame, value_col: str = 'revenue', date_col: str = 'date'):
        self.date_col = date_col
        self.value_col = value_col
        self.df = data.sort_values(by=self.date_col).reset_index(drop=True)
        self.execution_times: list[ExecutionTime] = []

    def _measure_execution(self, func: Callable, method_name: str, *args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        duration = time.perf_counter() - start_time
        
        self.execution_times.append(ExecutionTime(method_name, duration))
        return result

    def compare_methods(self, window: int = 7) -> float:
        pd_result = self._measure_execution(
            self.calculate_pandas_rolling, 
            "Pandas rolling", 
            window
        ).values
        
        np_result = self._measure_execution(
            self.calculate_numpy_rolling, 
            "NumPy rolling", 
            window
        )
        
        diff = np.abs(pd_result - np_result)
        max_diff = np.nanmax(diff)
        
        return max_diff

    def calculate_pandas_rolling(self, window: int = 7) -> pd.Series:
        return self.df[self.value_col].rolling(window=window).mean()

    def calculate_numpy_rolling(self, window: int = 7) -> np.ndarray:
        arr = self.df[self.value_col].values
        
        cumsum_vec = np.insert(arr, 0, 0).cumsum()
        rolling_sum = cumsum_vec[window:] - cumsum_vec[:-window]
        rolling_mean = rolling_sum / window
        
        result = np.full(len(arr), np.nan)
        result[window - 1:] = rolling_mean
        
        return result
    
    def print_execution_times(self) -> None:
        print("\nЧас виконання:")
        for exec_time in self.execution_times:
            print(f"  {exec_time}")

if __name__ == "__main__":
    np.random.seed(42)
    dates = pd.date_range(start="1900-01-01", periods=100000, freq="d")
    revenue = np.random.uniform(1000, 5000, size=100000)
    
    df_revenue = pd.DataFrame({'date': dates, 'revenue': revenue})
    
    print("Перші 10 рядків даних:")
    print(df_revenue.head(10))
    print(f"\nКількість записів: {len(df_revenue)}")
    print(f"Дата від {df_revenue['date'].min()} до {df_revenue['date'].max()}")
    
    analyzer = RevenueAnalyzer(df_revenue)
    
    window_size = 7
    max_difference = analyzer.compare_methods(window=window_size)
    
    print(f"\nМаксимальна абсолютна різниця між Pandas та NumPy (вікно {window_size}): {max_difference}")
    analyzer.print_execution_times()


Перші 10 рядків даних:
        date      revenue
0 1900-01-01  2498.160475
1 1900-01-02  4802.857226
2 1900-01-03  3927.975767
3 1900-01-04  3394.633937
4 1900-01-05  1624.074562
5 1900-01-06  1623.978081
6 1900-01-07  1232.334449
7 1900-01-08  4464.704583
8 1900-01-09  3404.460047
9 1900-01-10  3832.290311

Кількість записів: 100000
Дата від 1900-01-01 00:00:00 до 2173-10-15 00:00:00

Максимальна абсолютна різниця між Pandas та NumPy (вікно 7): 2.2645963326795027e-08

Час виконання:
  Pandas rolling: 0.003794s
  NumPy rolling: 0.001302s


#### Опис вхідних даних
Для тестування було згенеровано масштабний набір даних — рівно 100 000 записів. Кожен рядок імітує один день роботи бізнесу, містячи дату (у нашому синтетичному прикладі дати розтягнулися з 1 січня 1900 року аж до кінця 2173 року) та випадкову суму щоденної виручки в діапазоні від 1000 до 5000. Такий великий обсяг "сирих" даних дозволяє створити умови, наближені до реальних аналітичних завдань, щоб по-справжньому відчути різницю в продуктивності алгоритмів.

#### Результат абсолютної різниці (максимальна похибка)
Алгоритми видали практично ідентичний результат: максимальна абсолютна різниця між ними склала мізерні 2.26e-08 (це 0.0000000226). Ця крихітна відмінність не є помилкою логіки; вона виникає виключно через архітектурні особливості процесора під час роботи з дробовими числами (так звана арифметика з рухомою комою). На практиці ці алгоритми можна вважати математично рівноцінними, адже така похибка жодним чином не впливає на бізнес-рішення.

#### Порівняння часу виконання
Час виконання: Pandas rolling: 0.002042s NumPy rolling: 0.000798s 

Саме тут розкривається головна інтрига: класичний метод Pandas впорався із завданням за 0.002042 секунди, тоді як математичний трюк через NumPy — усього за 0.000798 секунди. Це означає, що алгоритм з кумулятивною сумою (cumsum) працює більш ніж у 2.5 рази швидше на масиві зі 100 000 рядків. Якби ми збільшили вікно з 7 днів до, скажімо, 365, Pandas сповільнився б ще більше, тоді як швидкість NumPy залишилася б майже незмінною, оскільки він робить лише одне віднімання незалежно від розміру вікна.

#### Де це корисно на практиці (на прикладі кав'ярні)
Уявіть, що ваша маленька кав'ярня виросла до величезної національної мережі, і тепер вам потрібно будувати щомісячні тренди (ковзне середнє за 30 днів) по сотнях точок за останні 10 років. Якщо ви використаєте класичний Pandas, ваш сервер може задуматися на кілька хвилин, поки перерахує мільйони рядків. А якщо застосуєте оптимізований підхід NumPy, фінансовий звіт для інвесторів на дашборді згенерується практично миттєво. В умовах великих даних (Big Data) такі хитрощі напряму економлять компанії гроші на оренді серверних потужностей.